# 07 — Multi-Panel Figures
**Goal:** Compose a full paper figure — multiple panels that tell a coherent story, with shared axes, panel labels (A/B/C), and callout annotations.

By the end of this notebook you will:
- Use `plt.subplots` for simple grids
- Use `GridSpec` for complex asymmetric layouts
- Share x or y axes across panels
- Add NeurIPS/ICML-style panel labels (bold A, B, C)
- Add arrow annotations and text callouts
- Compose a full 3-panel figure similar to what you'd submit

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

plt.style.use('research.mplstyle')

np.random.seed(3)

OKABE_ITO = ['#E69F00', '#56B4E9', '#009E73', '#F0E442', '#0072B2', '#D55E00', '#CC79A7']
OUR_COLOR = OKABE_ITO[4]   # blue

epochs = np.arange(1, 101)

def make_runs(n, decay, offset, noise):
    return np.array([
        (2.5 + np.random.randn()*0.05) * np.exp(-decay * epochs) + offset + noise * np.random.randn(100)
        for _ in range(n)
    ])

baseline_runs = make_runs(5, 0.06, 0.18, 0.07)
ours_runs     = make_runs(5, 0.09, 0.10, 0.07)

models_bar    = ['GPT-2', 'LLaMA', 'Mistral', 'Ours']
scores_bar    = [72.3, 75.8, 77.1, 81.2]
errors_bar    = [1.1, 0.8, 1.3, 0.7]

scores_dist   = np.concatenate([
    np.random.normal(78, 6, 200),
]).clip(0, 100)
baseline_dist = np.random.normal(71, 9, 200).clip(0, 100)

---
## Part 1 — Simple 1×N and N×M grids

`plt.subplots(nrows, ncols)` returns a 2D array of Axes. Use `figsize` to control the total canvas size.

**Exercise:** Create a 1×3 grid and make a different plot type in each panel (line, bar, scatter). Use `fig.tight_layout()` at the end.

In [ ]:
# TODO: fig, axes = plt.subplots(1, 3, figsize=(12, 3.5))
# axes[0]: training loss curve (ours_runs mean ± std)
# axes[1]: bar chart of scores_bar
# axes[2]: scatter of 100 random points
# Label each panel


---
## Part 2 — Shared axes

When panels share the same x or y range, use `sharex` or `sharey` to link their axes. This removes redundant tick labels and keeps scales consistent.

**Exercise:** Make a 2×1 figure (train loss on top, val loss on bottom) with shared x-axis. Remove x-tick labels from the top panel.

In [ ]:
# Simulate separate train and val loss
train_mean = ours_runs.mean(0)
train_std  = ours_runs.std(0)
val_runs   = make_runs(5, 0.08, 0.14, 0.09)
val_mean   = val_runs.mean(0)
val_std    = val_runs.std(0)

# TODO: fig, (ax_top, ax_bot) = plt.subplots(2, 1, figsize=(5, 5), sharex=True)
# ax_top: train loss (mean + band)
# ax_bot: val loss (mean + band)
# Remove x-tick labels from ax_top: plt.setp(ax_top.get_xticklabels(), visible=False)
# Add x-label only to ax_bot


---
## Part 3 — GridSpec for asymmetric layouts

`GridSpec` lets you create panels of different sizes — e.g., a wide main figure on the left and two stacked panels on the right.

```python
gs = gridspec.GridSpec(2, 2, width_ratios=[2, 1], hspace=0.4)
ax_main  = fig.add_subplot(gs[:, 0])    # spans both rows, left column
ax_top   = fig.add_subplot(gs[0, 1])    # top-right
ax_bot   = fig.add_subplot(gs[1, 1])    # bottom-right
```

**Exercise:** Create the layout above with:
- Left (wide): training curves for baseline vs ours
- Top-right: bar chart comparison
- Bottom-right: KDE of final epoch scores

In [ ]:
from scipy import stats as scipy_stats

# TODO: build the GridSpec layout described above
# fig = plt.figure(figsize=(10, 5))
# gs = gridspec.GridSpec(2, 2, width_ratios=[2, 1], hspace=0.5, wspace=0.4)

# ax_main = fig.add_subplot(gs[:, 0])
# ... fill each panel ...

# For the KDE panel, use final epoch scores:
# final_baseline = baseline_runs[:, -1]
# final_ours     = ours_runs[:, -1]


---
## Part 4 — Panel labels (A, B, C)

NeurIPS, ICML, and Nature figures use bold uppercase letters in the top-left corner of each panel.

Pattern:
```python
ax.text(-0.12, 1.05, 'A', transform=ax.transAxes,
        fontsize=13, fontweight='bold', va='top')
```

- `transform=ax.transAxes` — coordinates in (0,0)→(1,1) axes space, not data space
- `-0.12, 1.05` — slightly outside the top-left corner

**Exercise:** Add panel labels A, B, C to the three panels in Part 3.

In [ ]:
# TODO: copy the GridSpec figure from Part 3 and add panel labels A, B, C
# to ax_main, ax_top, and ax_bot respectively


---
## Part 5 — Callout annotations

Annotations guide the reader's eye to the key insight. Common uses:
- Pointing to the best result: `ax.annotate('Best', xy=(x, y), xytext=(...), arrowprops=...)`
- Shading a region of interest: `ax.axvspan(x0, x1, alpha=0.1, color='yellow')`
- Text boxes: `ax.text(..., bbox=dict(boxstyle='round', facecolor='white', edgecolor='gray'))`

**Exercise:** On a training curve, annotate the epoch of the minimum val loss with an arrow and label, and shade the first 20 epochs as a "warmup" region.

In [ ]:
mean_val = val_mean
best_epoch = epochs[np.argmin(mean_val)]
best_loss  = mean_val.min()

# TODO: plot val_mean as a line
# TODO: ax.axvspan(1, 20, alpha=0.08, color='orange', label='Warmup')
# TODO: ax.annotate(f'Best: {best_loss:.3f}',
#                   xy=(best_epoch, best_loss),
#                   xytext=(best_epoch + 10, best_loss + 0.1),
#                   arrowprops=dict(arrowstyle='->', color='#333333', lw=1.2),
#                   fontsize=9)


---
## Part 6 — Putting it all together: a complete paper figure

Build one cohesive figure with three panels that tell a single story:

**Story:** "Our model trains faster, achieves better benchmark scores, and has a tighter score distribution on agent tasks."

- **Panel A** (wide, left): training loss curves, baseline vs ours, with std bands
- **Panel B** (top-right): horizontal bar chart of benchmark scores
- **Panel C** (bottom-right): KDE of agent task score distributions

Add panel labels A/B/C. Add one annotation per panel pointing to the key insight.

In [ ]:
# TODO: compose the full 3-panel figure
# This is the capstone of the multi-panel notebook — bring together everything from Parts 1-5
# Use GridSpec with width_ratios=[2, 1]
# Panel A: curves with annotation at the point where 'Ours' pulls ahead
# Panel B: sorted horizontal bar chart, 'Ours' highlighted
# Panel C: KDE of scores_dist (ours) vs baseline_dist, filled
# All panels: panel labels, axis labels, legend


---
## Summary

| Concept | Pattern |
|---------|--------|
| Simple grid | `plt.subplots(nrows, ncols, figsize=...)` |
| Shared axis | `plt.subplots(..., sharex=True)` |
| Asymmetric layout | `GridSpec(rows, cols, width_ratios=[...])` |
| Panel label | `ax.text(-0.12, 1.05, 'A', transform=ax.transAxes, fontweight='bold')` |
| Arrow annotation | `ax.annotate(text, xy=data_point, xytext=label_pos, arrowprops=...)` |
| Region shading | `ax.axvspan(x0, x1, alpha=0.1, color=...)` |

**Next:** `08_export.ipynb` — saving figures at the right size and format for journal submission.